# ARC-AGI Pathfinder: Testing All Approaches

This notebook benchmarks multiple solving strategies on ARC tasks and reports detailed metrics.
Run on Kaggle (GPU T4/P100) or Colab, then share results for analysis.

**Approaches tested:**
1. DSL v2 â€” 77 primitives, depth 1-2 compositional search
2. DSL v3 â€” Heuristic solvers (pattern continuation, template stamping, object ops)
3. Neural TTT â€” Poincare ball encoder + FiLM decoder with test-time training
4. Hybrid â€” DSL first, neural fallback
5. Task-type routing â€” classify task, pick best solver

**Output:** JSON results file with per-task scores for offline analysis.

In [ ]:
# ============================================================
# Cell 1: Environment + Data Loading
# ============================================================
import subprocess, sys, os, json, time, copy, math, glob, ast, csv, re
import itertools
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Callable, Any, Set
from collections import defaultdict, Counter

import numpy as np

ON_KAGGLE = os.path.exists('/kaggle/input')
ON_COLAB = 'COLAB_GPU' in os.environ

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'batch-probe', 'kagglehub>=1.0.0'])

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Environment: {"Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "Local"}')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# --- batch-probe for GPU optimization ---
if DEVICE == 'cuda':
    from batch_probe import probe_batch_size, cached_probe, clear_cache
    print('batch-probe loaded â€” will auto-optimize batch sizes')
else:
    def probe_batch_size(*a, **kw): return 1
    def cached_probe(*a, **kw): return 1
    def clear_cache(): pass

# --- Data structures ---
@dataclass
class ARCPair:
    input: np.ndarray
    output: np.ndarray

@dataclass
class ARCTask:
    task_id: str
    train: List[ARCPair]
    test: List[ARCPair]

# ---- Loader 1: CSV (pshikk/arc-agi-csv-data) ----
def load_csv_tasks(csv_path):
    """id format: {task_id}_{train|test}[_{index}], grids as Python list literals."""
    tasks = defaultdict(lambda: {'train': [], 'test': []})
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rid = row['id']
            inp = np.array(ast.literal_eval(row['input']), dtype=int)
            out = np.array(ast.literal_eval(row['output']), dtype=int)
            parts = rid.split('_')
            task_id = parts[0]
            split = parts[1]  # 'train' or 'test'
            tasks[task_id][split].append(ARCPair(inp, out))
    return {tid: ARCTask(tid, s['train'], s['test']) for tid, s in tasks.items()}

# ---- Loader 2: Single JSON (arc-agi_training_challenges.json) ----
def load_json_challenges(chal_path, sol_path=None):
    with open(chal_path) as f: challenges = json.load(f)
    solutions = {}
    if sol_path and os.path.exists(sol_path):
        with open(sol_path) as f: solutions = json.load(f)
    tasks = {}
    for tid, td in challenges.items():
        train = [ARCPair(np.array(p['input']), np.array(p['output'])) for p in td['train']]
        test = []
        for i, p in enumerate(td['test']):
            inp = np.array(p['input'])
            out = np.array(solutions[tid][i]) if tid in solutions else (
                  np.array(p['output']) if 'output' in p else None)
            test.append(ARCPair(inp, out))
        tasks[tid] = ARCTask(tid, train, test)
    return tasks

# ---- Loader 3: Directory of individual JSONs ----
def load_json_dir(task_dir):
    tasks = {}
    for fp in sorted(Path(task_dir).glob('*.json')):
        with open(fp) as f: td = json.load(f)
        tid = fp.stem
        train = [ARCPair(np.array(p['input']), np.array(p['output'])) for p in td['train']]
        test = []
        for p in td['test']:
            inp = np.array(p['input'])
            out = np.array(p['output']) if 'output' in p else None
            test.append(ARCPair(inp, out))
        tasks[tid] = ARCTask(tid, train, test)
    return tasks

# ---- Auto-detect and load ----
def load_all_data():
    train_tasks, eval_tasks, test_tasks = {}, {}, {}

    # --- Strategy 1: kagglehub CSV dataset ---
    try:
        import kagglehub
        data_path = kagglehub.dataset_download('pshikk/arc-agi-csv-data')
        print(f'kagglehub data: {data_path}')
        for f in os.listdir(data_path):
            if not f.endswith('.csv'): continue
            fp = os.path.join(data_path, f)
            loaded = load_csv_tasks(fp)
            fl = f.lower()
            if 'train' in fl:
                train_tasks.update(loaded)
                print(f'  {f}: {len(loaded)} training tasks')
            elif 'eval' in fl:
                eval_tasks.update(loaded)
                print(f'  {f}: {len(loaded)} eval tasks')
            else:
                train_tasks.update(loaded)
                print(f'  {f}: {len(loaded)} tasks â†’ training')
    except Exception as e:
        print(f'kagglehub CSV load failed: {e}')

    # --- Strategy 2: Kaggle competition data (if attached) ---
    if ON_KAGGLE:
        for base in sorted(glob.glob('/kaggle/input/arc-*')):
            print(f'Scanning {base}...')
            # Single JSON files
            for name, bucket in [('training','train'),('evaluation','eval'),('test','test')]:
                for pattern in [f'arc-agi_{name}_challenges.json', f'{name}_challenges.json']:
                    chal = os.path.join(base, pattern)
                    if os.path.exists(chal):
                        sol = chal.replace('challenges', 'solutions')
                        loaded = load_json_challenges(chal, sol)
                        if bucket == 'train': train_tasks.update(loaded)
                        elif bucket == 'eval': eval_tasks.update(loaded)
                        else: test_tasks.update(loaded)
                        print(f'  {pattern}: {len(loaded)} {bucket} tasks')
            # Per-task JSON directories
            for subdir in os.listdir(base):
                dp = os.path.join(base, subdir)
                if os.path.isdir(dp) and len(list(Path(dp).glob('*.json'))) > 5:
                    loaded = load_json_dir(dp)
                    sl = subdir.lower()
                    if 'train' in sl: train_tasks.update(loaded)
                    elif 'eval' in sl: eval_tasks.update(loaded)
                    elif 'test' in sl: test_tasks.update(loaded)
                    else: train_tasks.update(loaded)
                    print(f'  {subdir}/: {len(loaded)} tasks')

    # --- Strategy 3: arckit fallback ---
    if not train_tasks:
        print('No data found, falling back to arckit...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arckit'])
        import arckit
        train_set, eval_set = arckit.load_data()
        def convert(tasks):
            r = {}
            for t in tasks:
                train = [ARCPair(np.array(i), np.array(o)) for i, o in t.train]
                test = [ARCPair(np.array(i), np.array(o)) for i, o in t.test]
                r[t.id] = ARCTask(t.id, train, test)
            return r
        train_tasks = convert(train_set)
        eval_tasks = convert(eval_set)

    return train_tasks, eval_tasks, test_tasks

train_tasks, eval_tasks, test_tasks = load_all_data()
print(f'\nFinal: Training={len(train_tasks)}, Eval={len(eval_tasks)}, Test={len(test_tasks)}')

# Sanity check
if train_tasks:
    sample = list(train_tasks.values())[0]
    print(f'Sample: {sample.task_id}, {len(sample.train)} train pairs, {len(sample.test)} test pairs')
    print(f'  Input: {sample.train[0].input.shape}, Output: {sample.train[0].output.shape}')

In [ ]:
# ============================================================
# Cell 2: Grid Utilities
# ============================================================
def grid_colors(g): return set(g.flatten().tolist())

def grid_bg(g):
    c = Counter(g.flatten().tolist())
    return c.most_common(1)[0][0]

def crop_to_content(g, bg=0):
    rows, cols = np.where(g != bg)
    if len(rows) == 0: return g.copy()
    return g[rows.min():rows.max()+1, cols.min():cols.max()+1].copy()

def extract_objects(g, bg=0, conn=4):
    h, w = g.shape
    visited = np.zeros_like(g, dtype=bool)
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)] if conn == 4 else \
           [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    objects = []
    for r in range(h):
        for c in range(w):
            if visited[r,c] or g[r,c] == bg: continue
            color = g[r,c]
            cells = []
            stack = [(r,c)]
            while stack:
                cr, cc = stack.pop()
                if 0<=cr<h and 0<=cc<w and not visited[cr,cc] and g[cr,cc]==color:
                    visited[cr,cc] = True
                    cells.append((cr,cc))
                    for dr,dc in nbrs: stack.append((cr+dr, cc+dc))
            rs = [r for r,c in cells]; cs = [c for r,c in cells]
            objects.append({'color': int(color), 'cells': cells,
                           'bbox': (min(rs),min(cs),max(rs)+1,max(cs)+1),
                           'size': len(cells)})
    return objects

def extract_objects_mc(g, bg=0):
    """Multi-color connected components."""
    h, w = g.shape
    visited = np.zeros_like(g, dtype=bool)
    objects = []
    for r in range(h):
        for c in range(w):
            if visited[r,c] or g[r,c] == bg: continue
            cells = []
            stack = [(r,c)]
            while stack:
                cr, cc = stack.pop()
                if 0<=cr<h and 0<=cc<w and not visited[cr,cc] and g[cr,cc]!=bg:
                    visited[cr,cc] = True
                    cells.append((cr,cc))
                    stack.extend([(cr-1,cc),(cr+1,cc),(cr,cc-1),(cr,cc+1)])
            rs = [r for r,c in cells]; cs = [c for r,c in cells]
            bbox = (min(rs),min(cs),max(rs)+1,max(cs)+1)
            sub = g[bbox[0]:bbox[2], bbox[1]:bbox[3]].copy()
            objects.append({'cells': cells, 'bbox': bbox, 'size': len(cells), 'subgrid': sub})
    return objects

def obj_subgrid(g, obj):
    r0,c0,r1,c1 = obj['bbox']
    return g[r0:r1, c0:c1].copy()

def analyze_task(task):
    info = {}
    out_shapes = set(); in_shapes = set()
    same = True
    for p in task.train:
        if p.input.shape != p.output.shape: same = False
        out_shapes.add(p.output.shape); in_shapes.add(p.input.shape)
    info['same_shape'] = same
    info['fixed_out'] = len(out_shapes) == 1
    info['out_shape'] = out_shapes.pop() if len(out_shapes)==1 else None
    ratios_h = set(); ratios_w = set()
    for p in task.train:
        ih,iw = p.input.shape; oh,ow = p.output.shape
        if ih>0: ratios_h.add(round(oh/ih, 4))
        if iw>0: ratios_w.add(round(ow/iw, 4))
    info['h_ratio'] = ratios_h.pop() if len(ratios_h)==1 else None
    info['w_ratio'] = ratios_w.pop() if len(ratios_w)==1 else None
    # Color analysis
    new_colors = set()
    for p in task.train:
        new_colors |= grid_colors(p.output) - grid_colors(p.input)
    info['introduces_new_colors'] = len(new_colors) > 0
    info['new_colors'] = new_colors
    return info
# --- Data Augmentation (D4 symmetry group: 8 geometric transforms) ---

def random_augment_pairs(pairs):
    """Apply a random D4 symmetry to all pairs consistently."""
    k = np.random.randint(8)
    if k == 0:
        return pairs
    ops = [
        lambda g: np.rot90(g, 1).copy(),
        lambda g: np.rot90(g, 2).copy(),
        lambda g: np.rot90(g, 3).copy(),
        lambda g: np.fliplr(g).copy(),
        lambda g: np.flipud(g).copy(),
        lambda g: g.T.copy(),
        lambda g: np.rot90(np.fliplr(g), 1).copy(),
    ]
    op = ops[k - 1]
    return [(op(i), op(o)) for i, o in pairs]

def majority_vote(pred_list):
    """Majority vote across multiple predictions per cell."""
    stacked = np.stack(pred_list)
    return np.apply_along_axis(
        lambda x: np.bincount(x, minlength=10).argmax(), 0, stacked
    ).astype(int)

print('Grid utilities loaded. (+ augmentation)')


In [ ]:
# ============================================================
# Cell 3: DSL v2 â€” 77 Atomic Primitives
# ============================================================

# --- Geometric ---
def identity(g): return g.copy()
def rot90(g): return np.rot90(g,1).copy()
def rot180(g): return np.rot90(g,2).copy()
def rot270(g): return np.rot90(g,3).copy()
def flip_h(g): return np.fliplr(g).copy()
def flip_v(g): return np.flipud(g).copy()
def transpose(g): return g.T.copy()
def transpose_anti(g): return np.rot90(np.fliplr(g)).copy()

# --- Cropping ---
def crop_bg(g): return crop_to_content(g, grid_bg(g))
def crop_bg0(g): return crop_to_content(g, 0)
def remove_border(g):
    if g.shape[0]<=2 or g.shape[1]<=2: return g.copy()
    return g[1:-1, 1:-1].copy()

# --- Tiling ---
def tile_2x2(g): return np.tile(g,(2,2))
def tile_3x3(g): return np.tile(g,(3,3))
def tile_2x1(g): return np.tile(g,(2,1))
def tile_1x2(g): return np.tile(g,(1,2))

# --- Scaling ---
def upscale_2x(g): return np.repeat(np.repeat(g,2,0),2,1)
def upscale_3x(g): return np.repeat(np.repeat(g,3,0),3,1)
def downscale_2x(g):
    h,w = g.shape
    if h%2 or w%2: return None
    return g[::2,::2].copy()
def downscale_3x(g):
    h,w = g.shape
    if h%3 or w%3: return None
    return g[::3,::3].copy()

# --- Color ---
def swap_bg_most(g):
    c = Counter(g.flatten().tolist()); mc = c.most_common(2)
    if len(mc)<2: return g.copy()
    bg,fg = mc[0][0],mc[1][0]; r=g.copy(); r[g==bg]=fg; r[g==fg]=bg; return r
def invert_colors(g):
    r=g.copy(); m=r>0; r[m]=9-r[m]; return r
def replace_with_most(g):
    bg=grid_bg(g); nb=g[g!=bg]
    if len(nb)==0: return g.copy()
    mc=Counter(nb.tolist()).most_common(1)[0][0]; r=g.copy(); r[r!=bg]=mc; return r
def remove_least_color(g):
    bg=grid_bg(g); nb=g[g!=bg]
    if len(nb)==0: return g.copy()
    lc=Counter(nb.tolist()).most_common()[-1][0]; r=g.copy(); r[r==lc]=bg; return r

# --- Mirror / symmetry ---
def mirror_h(g): return np.concatenate([g, np.fliplr(g)], axis=1)
def mirror_v(g): return np.concatenate([g, np.flipud(g)], axis=0)
def mirror_both(g):
    t = np.concatenate([g, np.fliplr(g)], axis=1)
    return np.concatenate([t, np.flipud(t)], axis=0)

# --- Border / fill ---
def fill_border(g):
    bg=grid_bg(g); nb=g[g!=bg]
    if len(nb)==0: return g.copy()
    fc=Counter(nb.tolist()).most_common(1)[0][0]
    r=g.copy(); r[0,:]=fc; r[-1,:]=fc; r[:,0]=fc; r[:,-1]=fc; return r
def hollow_interior(g):
    bg=grid_bg(g); r=np.full_like(g,bg)
    r[0,:]=g[0,:]; r[-1,:]=g[-1,:]; r[:,0]=g[:,0]; r[:,-1]=g[:,-1]; return r

# --- Gravity ---
def gravity_down(g):
    bg=grid_bg(g); r=np.full_like(g,bg)
    for c in range(g.shape[1]):
        v=g[:,c]; nz=v[v!=bg]
        if len(nz)>0: r[-len(nz):,c]=nz
    return r
def gravity_up(g):
    bg=grid_bg(g); r=np.full_like(g,bg)
    for c in range(g.shape[1]):
        v=g[:,c]; nz=v[v!=bg]
        if len(nz)>0: r[:len(nz),c]=nz
    return r
def gravity_left(g):
    bg=grid_bg(g); r=np.full_like(g,bg)
    for i in range(g.shape[0]):
        v=g[i]; nz=v[v!=bg]
        if len(nz)>0: r[i,:len(nz)]=nz
    return r
def gravity_right(g):
    bg=grid_bg(g); r=np.full_like(g,bg)
    for i in range(g.shape[0]):
        v=g[i]; nz=v[v!=bg]
        if len(nz)>0: r[i,-len(nz):]=nz
    return r

# --- Extract objects ---
def extract_largest(g):
    bg=grid_bg(g); objs=extract_objects(g,bg)
    if not objs: return g.copy()
    return obj_subgrid(g, max(objs, key=lambda o: o['size']))
def extract_smallest(g):
    bg=grid_bg(g); objs=extract_objects(g,bg)
    if not objs: return g.copy()
    return obj_subgrid(g, min(objs, key=lambda o: o['size']))
def extract_2nd_largest(g):
    bg=grid_bg(g); objs=extract_objects(g,bg)
    if len(objs)<2: return g.copy()
    objs.sort(key=lambda o: o['size'], reverse=True)
    return obj_subgrid(g, objs[1])
def extract_largest_mc(g):
    bg=grid_bg(g); objs=extract_objects_mc(g,bg)
    if not objs: return g.copy()
    return max(objs, key=lambda o: o['size'])['subgrid']

# --- Binary ---
def to_binary(g):
    bg=grid_bg(g); return (g!=bg).astype(int)

# --- Sort ---
def sort_rows(g): return np.sort(g, axis=1)
def sort_cols(g): return np.sort(g, axis=0)

# --- Halves ---
def top_half(g): return g[:g.shape[0]//2,:].copy()
def bottom_half(g): return g[g.shape[0]//2:,:].copy()
def left_half(g): return g[:,:g.shape[1]//2].copy()
def right_half(g): return g[:,g.shape[1]//2:].copy()
def top_left_quarter(g): h,w=g.shape; return g[:h//2,:w//2].copy()
def top_right_quarter(g): h,w=g.shape; return g[:h//2,w//2:].copy()
def bottom_left_quarter(g): h,w=g.shape; return g[h//2:,:w//2].copy()
def bottom_right_quarter(g): h,w=g.shape; return g[h//2:,w//2:].copy()

# --- Boolean half ops ---
def or_halves_h(g):
    h,w=g.shape
    if h%2: return None
    bg=grid_bg(g); t=g[:h//2,:].copy(); b=g[h//2:,:]; m=b!=bg; t[m]=b[m]; return t
def or_halves_v(g):
    h,w=g.shape
    if w%2: return None
    bg=grid_bg(g); l=g[:,:w//2].copy(); r=g[:,w//2:]; m=r!=bg; l[m]=r[m]; return l
def xor_halves_h(g):
    h,w=g.shape
    if h%2: return None
    bg=grid_bg(g); t=g[:h//2,:]; b=g[h//2:,:]
    r=np.full((h//2,w),bg,dtype=g.dtype)
    tm=t!=bg; bm=b!=bg; r[tm&~bm]=t[tm&~bm]; r[bm&~tm]=b[bm&~tm]; return r
def xor_halves_v(g):
    h,w=g.shape
    if w%2: return None
    bg=grid_bg(g); l=g[:,:w//2]; ri=g[:,w//2:]
    r=np.full((h,w//2),bg,dtype=g.dtype)
    lm=l!=bg; rm=ri!=bg; r[lm&~rm]=l[lm&~rm]; r[rm&~lm]=ri[rm&~lm]; return r
def and_halves_h(g):
    h,w=g.shape
    if h%2: return None
    bg=grid_bg(g); t=g[:h//2,:]; b=g[h//2:,:]; m=(t!=bg)&(b!=bg)
    r=np.full((h//2,w),bg,dtype=g.dtype); r[m]=t[m]; return r
def and_halves_v(g):
    h,w=g.shape
    if w%2: return None
    bg=grid_bg(g); l=g[:,:w//2]; ri=g[:,w//2:]; m=(l!=bg)&(ri!=bg)
    r=np.full((h,w//2),bg,dtype=g.dtype); r[m]=l[m]; return r

# --- Grid divider ops ---
def _find_h_divider(g):
    bg=grid_bg(g); h,w=g.shape
    for r in range(1,h-1):
        row=g[r,:]
        if len(set(row.tolist()))==1 and row[0]!=bg:
            return r, row[0]
    return None, None

def _find_v_divider(g):
    bg=grid_bg(g); h,w=g.shape
    for c in range(1,w-1):
        col=g[:,c]
        if len(set(col.tolist()))==1 and col[0]!=bg:
            return c, col[0]
    return None, None

def split_h_top(g):
    r,_=_find_h_divider(g)
    return g[:r,:].copy() if r else None
def split_h_bottom(g):
    r,_=_find_h_divider(g)
    return g[r+1:,:].copy() if r else None
def split_v_left(g):
    c,_=_find_v_divider(g)
    return g[:,:c].copy() if c else None
def split_v_right(g):
    c,_=_find_v_divider(g)
    return g[:,c+1:].copy() if c else None

def overlay_split_h(g):
    r,_=_find_h_divider(g)
    if r is None: return None
    t=g[:r,:]; b=g[r+1:,:]
    if t.shape!=b.shape: return None
    bg=grid_bg(g); res=b.copy(); m=t!=bg; res[m]=t[m]; return res
def overlay_split_v(g):
    c,_=_find_v_divider(g)
    if c is None: return None
    l=g[:,:c]; r=g[:,c+1:]
    if l.shape!=r.shape: return None
    bg=grid_bg(g); res=r.copy(); m=l!=bg; res[m]=l[m]; return res
def xor_split_h(g):
    r,_=_find_h_divider(g)
    if r is None: return None
    t=g[:r,:]; b=g[r+1:,:]
    if t.shape!=b.shape: return None
    bg=grid_bg(g); res=np.full_like(t,bg)
    tm=t!=bg; bm=b!=bg; x=tm^bm; res[x&tm]=t[x&tm]; res[x&bm]=b[x&bm]; return res
def and_split_h(g):
    r,_=_find_h_divider(g)
    if r is None: return None
    t=g[:r,:]; b=g[r+1:,:]
    if t.shape!=b.shape: return None
    bg=grid_bg(g); m=(t!=bg)&(b!=bg); res=np.full_like(t,bg); res[m]=t[m]; return res

# --- Symmetry completion ---
def complete_h_sym(g):
    h,w=g.shape; r=g.copy()
    for i in range(h):
        for j in range(w//2):
            mj=w-1-j
            if r[i,j]!=0 and r[i,mj]==0: r[i,mj]=r[i,j]
            elif r[i,mj]!=0 and r[i,j]==0: r[i,j]=r[i,mj]
    return r
def complete_v_sym(g):
    h,w=g.shape; r=g.copy()
    for i in range(h//2):
        mi=h-1-i
        for j in range(w):
            if r[i,j]!=0 and r[mi,j]==0: r[mi,j]=r[i,j]
            elif r[mi,j]!=0 and r[i,j]==0: r[i,j]=r[mi,j]
    return r
def complete_4sym(g): return complete_v_sym(complete_h_sym(g))

# --- Flood fill enclosed ---
def fill_enclosed(g):
    bg=grid_bg(g); h,w=g.shape
    visited=np.zeros_like(g,dtype=bool)
    queue=[]
    for r in range(h):
        for c in [0,w-1]:
            if g[r,c]==bg and not visited[r,c]: visited[r,c]=True; queue.append((r,c))
    for c in range(w):
        for r in [0,h-1]:
            if g[r,c]==bg and not visited[r,c]: visited[r,c]=True; queue.append((r,c))
    while queue:
        cr,cc=queue.pop(0)
        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr,nc=cr+dr,cc+dc
            if 0<=nr<h and 0<=nc<w and not visited[nr,nc] and g[nr,nc]==bg:
                visited[nr,nc]=True; queue.append((nr,nc))
    r=g.copy()
    for i in range(h):
        for j in range(w):
            if g[i,j]==bg and not visited[i,j]:
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
                    ni,nj=i+dr,j+dc
                    if 0<=ni<h and 0<=nj<w and g[ni,nj]!=bg:
                        r[i,j]=g[ni,nj]; break
    return r

# --- Period detection ---
def min_period_h(g):
    h,w=g.shape
    for p in range(1,h):
        if h%p: continue
        if np.array_equal(np.tile(g[:p,:],(h//p,1)), g): return g[:p,:].copy()
    return g.copy()
def min_period_v(g):
    h,w=g.shape
    for p in range(1,w):
        if w%p: continue
        if np.array_equal(np.tile(g[:,:p],(1,w//p)), g): return g[:,:p].copy()
    return g.copy()

# --- Row/col ops ---
def unique_rows(g):
    seen=[]; res=[]
    for r in range(g.shape[0]):
        row=tuple(g[r].tolist())
        if row not in seen: seen.append(row); res.append(g[r])
    return np.array(res) if res else g.copy()
def unique_cols(g): return unique_rows(g.T).T

# --- Diagonal ---
def main_diag(g): n=min(g.shape); return np.array([g[i,i] for i in range(n)]).reshape(1,-1)
def anti_diag(g): n=min(g.shape); w=g.shape[1]; return np.array([g[i,w-1-i] for i in range(n)]).reshape(1,-1)

# Build catalog
PRIMITIVES = {
    'identity': identity, 'rot90': rot90, 'rot180': rot180, 'rot270': rot270,
    'flip_h': flip_h, 'flip_v': flip_v, 'transpose': transpose, 'transpose_anti': transpose_anti,
    'crop_bg': crop_bg, 'crop_bg0': crop_bg0, 'remove_border': remove_border,
    'tile_2x2': tile_2x2, 'tile_3x3': tile_3x3, 'tile_2x1': tile_2x1, 'tile_1x2': tile_1x2,
    'upscale_2x': upscale_2x, 'upscale_3x': upscale_3x,
    'downscale_2x': downscale_2x, 'downscale_3x': downscale_3x,
    'swap_bg_most': swap_bg_most, 'invert_colors': invert_colors,
    'replace_with_most': replace_with_most, 'remove_least_color': remove_least_color,
    'mirror_h': mirror_h, 'mirror_v': mirror_v, 'mirror_both': mirror_both,
    'fill_border': fill_border, 'hollow_interior': hollow_interior,
    'gravity_down': gravity_down, 'gravity_up': gravity_up,
    'gravity_left': gravity_left, 'gravity_right': gravity_right,
    'extract_largest': extract_largest, 'extract_smallest': extract_smallest,
    'extract_2nd_largest': extract_2nd_largest, 'extract_largest_mc': extract_largest_mc,
    'to_binary': to_binary, 'sort_rows': sort_rows, 'sort_cols': sort_cols,
    'top_half': top_half, 'bottom_half': bottom_half,
    'left_half': left_half, 'right_half': right_half,
    'top_left_quarter': top_left_quarter, 'top_right_quarter': top_right_quarter,
    'bottom_left_quarter': bottom_left_quarter, 'bottom_right_quarter': bottom_right_quarter,
    'or_halves_h': or_halves_h, 'or_halves_v': or_halves_v,
    'xor_halves_h': xor_halves_h, 'xor_halves_v': xor_halves_v,
    'and_halves_h': and_halves_h, 'and_halves_v': and_halves_v,
    'split_h_top': split_h_top, 'split_h_bottom': split_h_bottom,
    'split_v_left': split_v_left, 'split_v_right': split_v_right,
    'overlay_split_h': overlay_split_h, 'overlay_split_v': overlay_split_v,
    'xor_split_h': xor_split_h, 'and_split_h': and_split_h,
    'complete_h_sym': complete_h_sym, 'complete_v_sym': complete_v_sym,
    'complete_4sym': complete_4sym, 'fill_enclosed': fill_enclosed,
    'min_period_h': min_period_h, 'min_period_v': min_period_v,
    'unique_rows': unique_rows, 'unique_cols': unique_cols,
    'main_diag': main_diag, 'anti_diag': anti_diag,
}

print(f'DSL v2: {len(PRIMITIVES)} primitives')

In [ ]:
# ============================================================
# Cell 4: Task-Specific Primitives + Search Engine
# ============================================================

def make_task_fns(task):
    fns = {}
    # --- Color replacement ---
    all_colors = set()
    for p in task.train:
        all_colors |= grid_colors(p.input) | grid_colors(p.output)
    for src in sorted(all_colors):
        for dst in sorted(all_colors):
            if src == dst: continue
            def mk(s,d):
                def fn(g): r=g.copy(); r[g==s]=d; return r
                return fn
            fns[f'recolor_{src}_{dst}'] = mk(src, dst)

    # --- Learned color map ---
    if all(p.input.shape == p.output.shape for p in task.train):
        mappings = defaultdict(set)
        for p in task.train:
            for i in range(p.input.shape[0]):
                for j in range(p.input.shape[1]):
                    mappings[int(p.input[i,j])].add(int(p.output[i,j]))
        if all(len(v)==1 for v in mappings.values()):
            cmap = {k: v.pop() for k,v in mappings.items()}
            if any(k != v for k,v in cmap.items()):  # non-identity
                def color_map(g, cm=dict(cmap)):
                    r = g.copy()
                    for s,d in cm.items(): r[g==s]=d
                    return r
                fns['learned_color_map'] = color_map

    # --- Fixed output ---
    outputs = [p.output for p in task.train]
    if all(np.array_equal(outputs[0], o) for o in outputs):
        fixed = outputs[0].copy()
        fns['fixed_output'] = lambda g, f=fixed: f.copy()

    # --- Size-based crops ---
    p0 = task.train[0]
    oh, ow = p0.output.shape
    ih, iw = p0.input.shape
    if oh <= ih and ow <= iw:
        fns[f'crop_tl_{oh}x{ow}'] = lambda g, h=oh, w=ow: g[:h,:w].copy()
        fns[f'crop_br_{oh}x{ow}'] = lambda g, h=oh, w=ow: g[-h:,-w:].copy()
        fns[f'crop_center_{oh}x{ow}'] = lambda g, h=oh, w=ow: g[(g.shape[0]-h)//2:(g.shape[0]-h)//2+h, (g.shape[1]-w)//2:(g.shape[1]-w)//2+w].copy()

    return fns

def apply_prog(g, prog):
    r = g
    for fn in prog:
        try:
            r = fn(r)
            if r is None or not isinstance(r, np.ndarray): return None
            if r.size == 0 or r.ndim != 2: return None
            if max(r.shape) > 30: return None
        except: return None
    return r

def verify_prog(task, prog):
    for p in task.train:
        r = apply_prog(p.input, prog)
        if r is None or not np.array_equal(r, p.output): return False
    return True

def verify_test(task, prog):
    for p in task.test:
        if p.output is None: return False
        r = apply_prog(p.input, prog)
        if r is None or not np.array_equal(r, p.output): return False
    return True

def search_dsl(task, max_depth=2, time_limit=30.0):
    t0 = time.time()
    all_p = dict(PRIMITIVES)
    all_p.update(make_task_fns(task))
    names = list(all_p.keys())
    fns = [all_p[n] for n in names]
    n = len(names)

    # Depth 1
    for i in range(n):
        if time.time()-t0 > time_limit: return None
        if verify_prog(task, [fns[i]]): return [names[i]], [fns[i]]
    if max_depth < 2: return None

    # Depth 2
    for i in range(n):
        if time.time()-t0 > time_limit: return None
        r1 = apply_prog(task.train[0].input, [fns[i]])
        if r1 is None: continue
        for j in range(n):
            if time.time()-t0 > time_limit: return None
            if verify_prog(task, [fns[i], fns[j]]):
                return [names[i], names[j]], [fns[i], fns[j]]
    return None

print('Search engine loaded.')

In [ ]:
# ============================================================
# Cell 5: DSL v3 â€” Heuristic Solvers
# ============================================================
# These go beyond fixed primitives to use task-specific reasoning.

def heuristic_pattern_continuation(task):
    """If the output is same-shape and matches input except some cells
    are filled in, try to learn the fill rule."""
    if not all(p.input.shape == p.output.shape for p in task.train):
        return None
    # Check: output = input with some 0s replaced
    for p in task.train:
        diff = (p.input != p.output)
        # Where they differ, input should be bg
        bg = grid_bg(p.input)
        if not np.all(p.input[diff] == bg):
            return None  # Input has non-bg cells that changed

    # Try: for each row, find the repeating pattern and extend it
    def extend_row_pattern(row, bg):
        """Try to find a period in non-bg positions and extend."""
        non_bg_pos = [i for i, v in enumerate(row) if v != bg]
        if len(non_bg_pos) < 2:
            return row.copy()
        # Find spacing
        gaps = [non_bg_pos[i+1] - non_bg_pos[i] for i in range(len(non_bg_pos)-1)]
        if len(set(gaps)) != 1:
            return row.copy()
        period = gaps[0]
        color = row[non_bg_pos[0]]
        result = row.copy()
        # Extend forward
        pos = non_bg_pos[-1] + period
        while pos < len(row):
            result[pos] = color
            pos += period
        # Extend backward
        pos = non_bg_pos[0] - period
        while pos >= 0:
            result[pos] = color
            pos -= period
        return result

    def solve(g):
        bg = grid_bg(g)
        r = g.copy()
        for i in range(r.shape[0]):
            r[i] = extend_row_pattern(r[i], bg)
        return r

    # Verify on training
    if verify_prog(task, [solve]):
        return 'pattern_cont_rows', [solve]

    # Try column-wise
    def solve_cols(g):
        return solve(g.T).T
    if verify_prog(task, [solve_cols]):
        return 'pattern_cont_cols', [solve_cols]

    return None


def heuristic_color_mapping_complex(task):
    """Learn per-cell color mapping based on neighbor context."""
    if not all(p.input.shape == p.output.shape for p in task.train):
        return None

    # For each cell that changes, look at its neighborhood
    # Try: majority neighbor color determines new color
    bg = grid_bg(task.train[0].input)

    def neighbor_majority(g, r, c, bg):
        h, w = g.shape
        counts = Counter()
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0: continue
                nr, nc = r+dr, c+dc
                if 0 <= nr < h and 0 <= nc < w and g[nr, nc] != bg:
                    counts[g[nr, nc]] += 1
        return counts.most_common(1)[0][0] if counts else bg

    def solve(g):
        r = g.copy()
        bg_val = grid_bg(g)
        for i in range(g.shape[0]):
            for j in range(g.shape[1]):
                if g[i, j] == bg_val:
                    # Check if enclosed
                    nm = neighbor_majority(g, i, j, bg_val)
                    # Only fill if surrounded on 3+ sides
                    adj_count = 0
                    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                        ni, nj = i+dr, j+dc
                        if 0<=ni<g.shape[0] and 0<=nj<g.shape[1] and g[ni,nj]!=bg_val:
                            adj_count += 1
                        elif not (0<=ni<g.shape[0] and 0<=nj<g.shape[1]):
                            pass  # border
                    if adj_count >= 3:
                        r[i, j] = nm
        return r

    if verify_prog(task, [solve]):
        return 'neighbor_fill', [solve]
    return None


def heuristic_object_copy(task):
    """Detect if output copies/moves objects to new locations."""
    if not all(p.input.shape == p.output.shape for p in task.train):
        return None

    bg = grid_bg(task.train[0].input)

    # Check if output has same objects as input but in different positions
    # Or if output has objects from input plus copies

    # Simple case: input has objects, output has the same objects tiled/reflected within grid
    # Try: take each object, tile it across the grid
    for p in task.train:
        in_objs = extract_objects(p.input, bg)
        out_objs = extract_objects(p.output, bg)
        if len(out_objs) > len(in_objs) * 1.5:  # output has more objects
            pass  # Could be object replication
        else:
            return None

    return None  # Placeholder for future development


def heuristic_majority_vote(task):
    """For same-shape tasks: for each cell, take the most common
    non-bg value at that position across all train outputs."""
    if not all(p.input.shape == p.output.shape for p in task.train):
        return None
    # This is a meta-heuristic: can we predict output just from input?
    # Try: output = input except where it differs, learn the rule
    return None  # Placeholder


def heuristic_row_col_repeat(task):
    """Detect if output is input with certain rows/cols repeated to fill."""
    if not all(p.input.shape == p.output.shape for p in task.train):
        return None

    bg = grid_bg(task.train[0].input)

    # Check: each output row is a tiling of some segment of the input row
    def try_tile_rows(g):
        """For each row, if it has a non-bg segment, tile it to fill the row."""
        bg_val = grid_bg(g)
        r = g.copy()
        for i in range(g.shape[0]):
            row = g[i]
            non_bg = np.where(row != bg_val)[0]
            if len(non_bg) == 0: continue
            # Get the non-bg segment
            start, end = non_bg[0], non_bg[-1] + 1
            segment = row[start:end]
            seg_len = len(segment)
            if seg_len == 0: continue
            # Tile this segment across the full row
            full = np.tile(segment, (g.shape[1] // seg_len) + 2)[:g.shape[1]]
            r[i] = full
        return r

    if verify_prog(task, [try_tile_rows]):
        return 'tile_rows', [try_tile_rows]

    def try_tile_cols(g):
        return try_tile_rows(g.T).T

    if verify_prog(task, [try_tile_cols]):
        return 'tile_cols', [try_tile_cols]

    return None


def heuristic_keep_color(task):
    """Output keeps only certain colors from input, rest become bg."""
    if not all(p.input.shape == p.output.shape for p in task.train):
        return None

    bg = grid_bg(task.train[0].input)
    # Find which colors are kept
    kept = None
    for p in task.train:
        out_colors = grid_colors(p.output) - {bg}
        if kept is None:
            kept = out_colors
        else:
            kept &= out_colors
    if kept is None or len(kept) == 0:
        return None

    def solve(g, keep=frozenset(kept), bg_val=bg):
        r = np.full_like(g, bg_val)
        for c in keep:
            r[g == c] = c
        return r

    if verify_prog(task, [solve]):
        return f'keep_colors_{sorted(kept)}', [solve]
    return None


def heuristic_mask_overlay(task):
    """Use one object as a mask/template over another."""
    bg = grid_bg(task.train[0].input)
    # Find if there are exactly 2 distinct objects/regions
    p = task.train[0]
    objs = extract_objects_mc(p.input, bg)
    if len(objs) != 2:
        return None

    o1, o2 = objs[0], objs[1]
    s1, s2 = o1['subgrid'], o2['subgrid']
    # Try overlaying smaller on larger
    if s1.shape == s2.shape:
        def overlay_12(g):
            bg_val = grid_bg(g)
            objs_g = extract_objects_mc(g, bg_val)
            if len(objs_g) != 2: return None
            a, b = objs_g[0]['subgrid'], objs_g[1]['subgrid']
            if a.shape != b.shape: return None
            r = b.copy()
            m = a != bg_val
            r[m] = a[m]
            return r

        if verify_prog(task, [overlay_12]):
            return 'overlay_objs', [overlay_12]

        def overlay_21(g):
            bg_val = grid_bg(g)
            objs_g = extract_objects_mc(g, bg_val)
            if len(objs_g) != 2: return None
            a, b = objs_g[0]['subgrid'], objs_g[1]['subgrid']
            if a.shape != b.shape: return None
            r = a.copy()
            m = b != bg_val
            r[m] = b[m]
            return r

        if verify_prog(task, [overlay_21]):
            return 'overlay_objs_rev', [overlay_21]

    return None


HEURISTICS = [
    ('pattern_cont', heuristic_pattern_continuation),
    ('color_map_cx', heuristic_color_mapping_complex),
    ('row_col_repeat', heuristic_row_col_repeat),
    ('keep_color', heuristic_keep_color),
    ('mask_overlay', heuristic_mask_overlay),
]

def search_heuristic(task):
    for name, hfn in HEURISTICS:
        try:
            result = hfn(task)
            if result is not None:
                return result
        except Exception:
            continue
    return None

print(f'Heuristic solvers: {len(HEURISTICS)}')

In [ ]:
# ============================================================
# Cell 6: Neural Model (scaled up, ensemble, batch-probe)
# ============================================================
NC = 10; MG = 30

def g2t(g, dev='cpu'):
    a = np.array(g, dtype=np.int64); h,w = a.shape
    oh = np.zeros((NC,h,w), dtype=np.float32)
    for c in range(NC): oh[c] = (a==c).astype(np.float32)
    return torch.from_numpy(oh).to(dev)

def pad_t(t, sz=MG):
    c,h,w = t.shape
    if h>=sz and w>=sz: return t[:,:sz,:sz]
    p = torch.zeros(c,sz,sz, dtype=t.dtype, device=t.device)
    p[:,:h,:w] = t; return p

def mk_mask(h,w, sz=MG, dev='cpu'):
    m = torch.zeros(sz,sz, device=dev); m[:h,:w] = 1.0; return m

def batch_encode_grids(grids, dev):
    tensors, masks = [], []
    for g in grids:
        gl = g.tolist() if isinstance(g, np.ndarray) else g
        t = g2t(gl, dev)
        tensors.append(pad_t(t))
        h, w = (g.shape if isinstance(g, np.ndarray) else (len(g), len(g[0])))
        masks.append(mk_mask(h, w, dev=dev))
    return torch.stack(tensors), torch.stack(masks)

# --- Model components (scaled: zd=192, hid=384, hd=48) ---

class GEnc(nn.Module):
    def __init__(self, zd=192, hid=384):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(NC, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.GroupNorm(8, 128), nn.GELU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.GroupNorm(16, 256), nn.GELU(),
            nn.Conv2d(256, hid, 3, padding=1), nn.GroupNorm(16, hid), nn.GELU(),
        )
        self.ap = nn.Linear(hid, 1)
        self.proj = nn.Sequential(nn.Linear(hid, zd), nn.LayerNorm(zd))
    def forward(self, grid, mask):
        f = self.conv(grid); b,c,h,w = f.shape
        ff = f.view(b,c,h*w).permute(0,2,1)
        s = self.ap(ff).squeeze(-1); mf = mask.view(b,h*w)
        s = s.masked_fill(mf==0, float('-inf'))
        wt = F.softmax(s, dim=-1).unsqueeze(-1)
        return self.proj((ff*wt).sum(dim=1))

class PEnc(nn.Module):
    def __init__(self, zd=192):
        super().__init__()
        self.ge = GEnc(zd=zd)
        self.pp = nn.Sequential(
            nn.Linear(zd*3, zd), nn.LayerNorm(zd), nn.GELU(), nn.Linear(zd, zd))
    def forward(self, ig, im, og, om):
        zi = self.ge(ig, im); zo = self.ge(og, om)
        return self.pp(torch.cat([zi, zo, zo-zi], dim=-1))

class PBall:
    @staticmethod
    def project(x, c=1.0, mn=0.95):
        n = x.norm(dim=-1, keepdim=True); mr = mn/(c**0.5)
        return torch.where(n > mr, x*mr/n, x)

class HREnc(nn.Module):
    def __init__(self, zd=192, hd=48, c=1.0):
        super().__init__()
        self.c = c
        self.proj = nn.Sequential(nn.Linear(zd, zd), nn.GELU(), nn.Linear(zd, hd))
    def forward(self, z): return PBall.project(self.proj(z), self.c)

class RCond(nn.Module):
    def __init__(self, zd, nc):
        super().__init__()
        self.g = nn.Linear(zd, nc); self.b = nn.Linear(zd, nc)
    def forward(self, feat, zr):
        return self.g(zr).unsqueeze(-1).unsqueeze(-1) * feat + self.b(zr).unsqueeze(-1).unsqueeze(-1)

class GDec(nn.Module):
    def __init__(self, zd=192, hid=384):
        super().__init__()
        self.zd = zd
        self.ic = nn.Sequential(
            nn.Conv2d(NC, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.GELU())
        self.comb = nn.Conv2d(64 + zd, hid, 1)
        self.layers = nn.ModuleList([nn.Sequential(
            nn.Conv2d(hid, hid, 3, padding=1), nn.GroupNorm(16, hid), nn.GELU()
        ) for _ in range(6)])
        self.conds = nn.ModuleList([RCond(zd, hid) for _ in range(6)])
        self.out = nn.Conv2d(hid, NC, 1)
    def forward(self, zr, tg, tm):
        b = zr.shape[0]
        inf = self.ic(tg)
        zs = zr.unsqueeze(-1).unsqueeze(-1).expand(b, self.zd, MG, MG)
        h = self.comb(torch.cat([inf, zs], dim=1))
        for l, c in zip(self.layers, self.conds):
            h = h + c(l(h), zr)   # residual connections
        return self.out(h) * tm.unsqueeze(1)

# --- Probe the combined encoder+decoder pass ---

class _TrainProbe(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.pe = model.pe; self.re = model.re; self.ra = model.ra
        self.rz = model.rz; self.dec = model.dec; self.zd = model.zd
    def forward(self, ig, im, og, om):
        z = self.pe(ig, im, og, om)
        h = self.re(z)
        w = self.ra(h)
        h_agg = PBall.project((w * h).sum(dim=0, keepdim=True))
        zr = self.rz(h_agg).expand(ig.shape[0], -1)
        logits = self.dec(zr, ig, im)
        targets = og.argmax(dim=1).long()
        loss = F.cross_entropy(logits, targets, reduction='none')
        return (loss * om).sum() / om.sum().clamp(min=1)

_GPU_MAX_PAIRS = {'val': 16, 'probed': False}

def probe_gpu_limits(model, dev):
    if dev != 'cuda' or _GPU_MAX_PAIRS['probed']:
        return
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'batch-probe: {vram:.1f} GB VRAM, probing combined enc+dec...')
    tp = _TrainProbe(model).to(dev)
    try:
        _GPU_MAX_PAIRS['val'] = probe_batch_size(
            tp,
            lambda bs: {'ig': torch.zeros(bs,NC,MG,MG,device=dev),
                        'im': torch.ones(bs,MG,MG,device=dev),
                        'og': torch.zeros(bs,NC,MG,MG,device=dev),
                        'om': torch.ones(bs,MG,MG,device=dev)},
            mode='train', low=4, high=2048, headroom=0.15, verbose=True)
        print(f'  => Max pairs per pass: {_GPU_MAX_PAIRS["val"]}')
    except Exception as e:
        print(f'  Probe failed ({e}), default={_GPU_MAX_PAIRS["val"]}')
    del tp; torch.cuda.empty_cache()
    _GPU_MAX_PAIRS['probed'] = True


def multi_task_train_step(model, task_pairs_list, dev):
    """Process multiple tasks in ONE batched GPU pass."""
    all_in, all_out = [], []
    task_slices = []
    offset = 0
    for pairs in task_pairs_list:
        n = len(pairs)
        all_in.extend([p[0] for p in pairs])
        all_out.extend([p[1] for p in pairs])
        task_slices.append((offset, offset + n))
        offset += n

    ig_all, im_all = batch_encode_grids(all_in, dev)
    og_all, om_all = batch_encode_grids(all_out, dev)
    z_all = model.pe(ig_all, im_all, og_all, om_all)
    h_all = model.re(z_all)

    zr_list = []
    for start, end in task_slices:
        N = end - start
        h_task = h_all[start:end]
        if N < 2:
            w = model.ra(h_task)
            h_agg = PBall.project((w * h_task).sum(dim=0, keepdim=True))
            zr_list.append(model.rz(h_agg).squeeze(0))
        else:
            for idx in range(N):
                mask = torch.ones(N, dtype=torch.bool, device=dev); mask[idx] = False
                h_ctx = h_task[mask]
                w_ctx = model.ra(h_ctx)
                h_agg = PBall.project((w_ctx * h_ctx).sum(dim=0, keepdim=True))
                zr_list.append(model.rz(h_agg).squeeze(0))
    zr_batch = torch.stack(zr_list)

    logits = model.dec(zr_batch, ig_all, im_all)
    targets = og_all.argmax(dim=1).long()
    loss = F.cross_entropy(logits, targets, reduction='none')
    loss = (loss * om_all).sum() / om_all.sum().clamp(min=1)
    return loss


ZD, HD, HID = 192, 48, 384  # model hyperparams (used by pretrain)

class NeuralSolver(nn.Module):
    def __init__(self, zd=ZD, hd=HD, hid=HID):
        super().__init__()
        self.pe = PEnc(zd=zd)
        self.re = HREnc(zd=zd, hd=hd)
        self.dec = GDec(zd=zd, hid=hid)
        self.ra = nn.Sequential(nn.Linear(hd, 1), nn.Softmax(dim=0))
        self.rz = nn.Sequential(
            nn.Linear(hd, zd), nn.LayerNorm(zd), nn.GELU(), nn.Linear(zd, zd))
        self.zd = zd

    def infer_rule_batched(self, pairs, dev):
        in_grids = [p[0] for p in pairs]
        out_grids = [p[1] for p in pairs]
        ig_b, im_b = batch_encode_grids(in_grids, dev)
        og_b, om_b = batch_encode_grids(out_grids, dev)
        z_pairs = self.pe(ig_b, im_b, og_b, om_b)
        h_rules = self.re(z_pairs)
        weights = self.ra(h_rules)
        h_agg = PBall.project((weights * h_rules).sum(dim=0, keepdim=True))
        return self.rz(h_agg)

    def predict_batched(self, zr, test_inputs, dev):
        tg_b, tm_b = batch_encode_grids(test_inputs, dev)
        zr_exp = zr.expand(tg_b.shape[0], -1)
        logits = self.dec(zr_exp, tg_b, tm_b)
        preds = logits.argmax(dim=1)
        return [preds[i, :ti.shape[0], :ti.shape[1]].cpu().numpy().astype(int)
                for i, ti in enumerate(test_inputs)]

    def train_step_batched(self, pairs, dev):
        return multi_task_train_step(self, [pairs], dev)

    def refine(self, pairs, dev, steps=200, lr=1e-4):
        params = list(self.dec.parameters()) + list(self.ra.parameters()) + list(self.rz.parameters())
        opt = optim.Adam(params, lr=lr)
        self.train()
        for s in range(steps):
            opt.zero_grad()
            aug = random_augment_pairs(pairs)
            loss = self.train_step_batched(aug, dev)
            loss.backward()
            opt.step()
        self.eval()
        with torch.no_grad():
            return self.infer_rule_batched(pairs, dev)

    def solve(self, train_pairs, test_inputs, dev, refine_flag=True, steps=200):
        self.to(dev)
        state = copy.deepcopy(self.state_dict())
        try:
            if refine_flag:
                zr = self.refine(train_pairs, dev, steps=steps)
            else:
                self.eval()
                with torch.no_grad():
                    zr = self.infer_rule_batched(train_pairs, dev)
            self.eval()
            with torch.no_grad():
                preds = self.predict_batched(zr, test_inputs, dev)
            results = [[p, p.copy()] for p in preds]
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache()
                results = [[np.zeros((2,2), dtype=int)]*2 for _ in test_inputs]
            else:
                raise
        finally:
            self.load_state_dict(state)
        return results

    def solve_ensemble(self, train_pairs, test_inputs, dev, n_runs=5, steps=200):
        """Run TTT n_runs times with different seeds, majority vote per cell."""
        self.to(dev)
        state = copy.deepcopy(self.state_dict())
        all_preds = []

        for run in range(n_runs):
            self.load_state_dict(copy.deepcopy(state))
            torch.manual_seed(run * 137 + 42)
            try:
                zr = self.refine(train_pairs, dev, steps=steps)
                self.eval()
                with torch.no_grad():
                    preds = self.predict_batched(zr, test_inputs, dev)
                all_preds.append(preds)
            except RuntimeError as e:
                if 'out of memory' in str(e).lower():
                    torch.cuda.empty_cache()
                continue

        self.load_state_dict(state)

        if not all_preds:
            return [[np.zeros((2,2), dtype=int)]*2 for _ in test_inputs]

        results = []
        for ti_idx in range(len(test_inputs)):
            votes = [all_preds[r][ti_idx] for r in range(len(all_preds))]
            voted = majority_vote(votes)
            results.append([voted, voted.copy()])
        return results

nparams = sum(p.numel() for p in NeuralSolver().parameters())
print(f'Neural model: {nparams/1e6:.1f}M params (zd={ZD}, hd={HD}, hid={HID})')
print(f'  4 encoder conv layers, 6 decoder conv layers (with residuals)')
print(f'  Ensemble TTT, D4 augmentation during refine')

In [ ]:
# ============================================================
# Cell 7: Pre-train Neural Model (augmented, auto-scaled)
# ============================================================

def pretrain(model, tasks, dev, epochs=30, lr=3e-4, max_per_epoch=400):
    model.to(dev); model.train()
    tl = list(tasks.values())

    probe_gpu_limits(model, dev)
    max_pairs = _GPU_MAX_PAIRS['val']

    avg_pairs = sum(len(t.train) for t in tl) / max(len(tl), 1)
    tasks_per_batch = max(1, int(max_pairs / avg_pairs))
    steps_per_epoch = max(1, max_per_epoch // tasks_per_batch)

    target_total_steps = 600
    total_steps_at_current = steps_per_epoch * epochs
    if total_steps_at_current < target_total_steps:
        epochs = max(epochs, target_total_steps // steps_per_epoch)

    lr_scale = (tasks_per_batch / 8) ** 0.5
    effective_lr = lr * lr_scale

    print(f'max_pairs={max_pairs}, avg_pairs/task={avg_pairs:.1f}')
    print(f'tasks_per_batch={tasks_per_batch}, steps/epoch={steps_per_epoch}')
    print(f'epochs={epochs}, total_steps~{steps_per_epoch * epochs}')
    print(f'LR: {lr} x {lr_scale:.1f} = {effective_lr:.5f}')
    print(f'Data augmentation: D4 symmetry (8x effective data)')

    opt = optim.Adam(model.parameters(), lr=effective_lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    for ep in range(epochs):
        np.random.shuffle(tl)
        eloss = 0; n = min(len(tl), max_per_epoch); steps_done = 0

        for i in range(0, n, tasks_per_batch):
            opt.zero_grad()
            batch_tasks = []
            for j in range(i, min(i + tasks_per_batch, n)):
                pairs = [(p.input, p.output) for p in tl[j].train]
                if len(pairs) < 2: continue
                # Random D4 augmentation per task per epoch
                pairs = random_augment_pairs(pairs)
                batch_tasks.append(pairs)
            if not batch_tasks: continue
            try:
                loss = multi_task_train_step(model, batch_tasks, dev)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                eloss += loss.item()
                steps_done += 1
            except RuntimeError as e:
                if 'out of memory' in str(e).lower():
                    torch.cuda.empty_cache()
                    tasks_per_batch = max(1, tasks_per_batch // 2)
                    print(f'  OOM! Reduced to {tasks_per_batch} tasks/batch')
                    continue
                raise

        sched.step()
        if (ep+1) % 10 == 0 or ep == 0:
            avg = eloss / max(steps_done, 1)
            mem = torch.cuda.max_memory_allocated() / 1e9 if dev == 'cuda' else 0
            print(f'  Epoch {ep+1}/{epochs} | Loss: {avg:.4f} | Steps: {steps_done} | VRAM: {mem:.1f}GB')

    model.eval()
    return model

print('Pre-training on', DEVICE, '...')
neural_model = NeuralSolver()

if DEVICE == 'cuda':
    EPOCHS = 30; MAX_PER = 400
else:
    EPOCHS = 5; MAX_PER = 50

neural_model = pretrain(neural_model, train_tasks, DEVICE, epochs=EPOCHS, max_per_epoch=MAX_PER)
print('Pre-training complete.')

In [ ]:
# ============================================================
# Cell 8: Evaluation Framework (ensemble + augmented TTT)
# ============================================================

def cell_accuracy(pred, target):
    if pred is None or target is None: return 0.0
    p = np.array(pred); t = np.array(target)
    if p.shape != t.shape: return 0.0
    return float(np.mean(p == t))

def evaluate_all_approaches(tasks, model, dev, max_tasks=None,
                            dsl_time=30.0, ttt_steps=200, n_ensemble=5):
    task_list = list(tasks.values())
    if max_tasks: task_list = task_list[:max_tasks]
    n = len(task_list)

    approaches = {
        'dsl_v2': {'correct': 0, 'total_cell_acc': 0.0},
        'heuristic': {'correct': 0, 'total_cell_acc': 0.0},
        'neural_no_ttt': {'correct': 0, 'total_cell_acc': 0.0},
        'neural_ttt': {'correct': 0, 'total_cell_acc': 0.0},
        'neural_ensemble': {'correct': 0, 'total_cell_acc': 0.0},
        'hybrid_best': {'correct': 0, 'total_cell_acc': 0.0},
    }
    per_task = []

    for ti, task in enumerate(task_list):
        if (ti+1) % 20 == 0 or ti == 0:
            print(f'  Task {ti+1}/{n}...')

        test_pairs = task.test
        if not test_pairs or test_pairs[0].output is None:
            continue

        train_pairs = [(p.input, p.output) for p in task.train]
        test_inputs = [p.input for p in test_pairs]
        test_outputs = [p.output for p in test_pairs]
        same_shape = all(p.input.shape == p.output.shape for p in task.train)

        result = {
            'task_id': task.task_id,
            'same_shape': same_shape,
            'n_train': len(task.train),
            'n_test': len(test_pairs),
        }

        # --- DSL v2 ---
        t0 = time.time()
        dsl_solved = False; dsl_prog = None; dsl_preds = []
        try:
            sr = search_dsl(task, time_limit=dsl_time)
            if sr is not None:
                prog_names, prog_fns = sr
                dsl_prog = ' -> '.join(prog_names)
                for ti_g in test_inputs:
                    p = apply_prog(ti_g, prog_fns)
                    dsl_preds.append(p)
                dsl_solved = all(
                    np.array_equal(p, t) for p, t in zip(dsl_preds, test_outputs)
                    if p is not None)
        except Exception:
            pass
        dsl_t = time.time() - t0
        dsl_ca = np.mean([cell_accuracy(p, t) for p, t in zip(dsl_preds, test_outputs)]) if dsl_preds else 0.0
        result['dsl_v2'] = {'solved': dsl_solved, 'program': dsl_prog,
                            'cell_acc': float(dsl_ca), 'time': dsl_t}

        # --- Heuristic ---
        t0 = time.time()
        heur_solved = False; heur_name = None; heur_preds = []
        try:
            hr = search_heuristic(task)
            if hr is not None:
                heur_name, heur_fns = hr
                for ti_g in test_inputs:
                    p = apply_prog(ti_g, heur_fns)
                    heur_preds.append(p)
                heur_solved = all(
                    np.array_equal(p, t) for p, t in zip(heur_preds, test_outputs)
                    if p is not None)
        except Exception:
            pass
        heur_t = time.time() - t0
        heur_ca = np.mean([cell_accuracy(p, t) for p, t in zip(heur_preds, test_outputs)]) if heur_preds else 0.0
        result['heuristic'] = {'solved': heur_solved, 'name': heur_name,
                               'cell_acc': float(heur_ca), 'time': heur_t}

        # --- Neural no-TTT ---
        t0 = time.time()
        nottt_solved = False; nottt_preds = []
        try:
            with torch.no_grad():
                preds = model.solve(train_pairs, test_inputs, dev, refine_flag=False)
            nottt_preds = [p[0] for p in preds]
            nottt_solved = all(
                np.array_equal(p, t) for p, t in zip(nottt_preds, test_outputs))
        except Exception:
            if dev == 'cuda': torch.cuda.empty_cache()
        nottt_t = time.time() - t0
        nottt_ca = np.mean([cell_accuracy(p, t) for p, t in zip(nottt_preds, test_outputs)]) if nottt_preds else 0.0
        result['neural_no_ttt'] = {'solved': nottt_solved,
                                    'cell_acc': float(nottt_ca), 'time': nottt_t}

        # --- Neural TTT (single run) ---
        t0 = time.time()
        ttt_solved = False; ttt_preds = []
        try:
            preds = model.solve(train_pairs, test_inputs, dev, refine_flag=True, steps=ttt_steps)
            ttt_preds = [p[0] for p in preds]
            ttt_solved = all(
                np.array_equal(p, t) for p, t in zip(ttt_preds, test_outputs))
        except Exception:
            if dev == 'cuda': torch.cuda.empty_cache()
        ttt_t = time.time() - t0
        ttt_ca = np.mean([cell_accuracy(p, t) for p, t in zip(ttt_preds, test_outputs)]) if ttt_preds else 0.0
        result['neural_ttt'] = {'solved': ttt_solved,
                                 'cell_acc': float(ttt_ca), 'time': ttt_t}

        # --- Neural Ensemble TTT ---
        t0 = time.time()
        ens_solved = False; ens_preds = []
        if n_ensemble > 1:
            try:
                preds = model.solve_ensemble(train_pairs, test_inputs, dev,
                                              n_runs=n_ensemble, steps=ttt_steps)
                ens_preds = [p[0] for p in preds]
                ens_solved = all(
                    np.array_equal(p, t) for p, t in zip(ens_preds, test_outputs))
            except Exception:
                if dev == 'cuda': torch.cuda.empty_cache()
        ens_t = time.time() - t0
        ens_ca = np.mean([cell_accuracy(p, t) for p, t in zip(ens_preds, test_outputs)]) if ens_preds else 0.0
        result['neural_ensemble'] = {'solved': ens_solved,
                                      'cell_acc': float(ens_ca), 'time': ens_t}

        # --- Hybrid best ---
        hybrid_solved = dsl_solved or heur_solved or ttt_solved or ens_solved
        hybrid_ca = max(dsl_ca, heur_ca, nottt_ca, ttt_ca, ens_ca)
        result['hybrid_best'] = {'solved': hybrid_solved, 'cell_acc': float(hybrid_ca)}

        for key in approaches:
            if result[key]['solved']:
                approaches[key]['correct'] += 1
            approaches[key]['total_cell_acc'] += result[key].get('cell_acc', 0.0)

        per_task.append(result)

    for key in approaches:
        approaches[key]['rate'] = approaches[key]['correct'] / max(n, 1)
        approaches[key]['avg_cell_acc'] = approaches[key]['total_cell_acc'] / max(n, 1)

    return {
        'approaches': approaches,
        'per_task': per_task,
        'meta': {'n_tasks': n, 'ttt_steps': ttt_steps, 'n_ensemble': n_ensemble},
    }

print('Evaluation framework ready (ensemble + augmented TTT).')

In [ ]:
# ============================================================
# Cell 9: Run Evaluation on TRAINING Tasks (sample)
# ============================================================
print('='*60)
print('EVALUATING ON TRAINING TASKS (sample)')
print('='*60)

train_results = evaluate_all_approaches(
    train_tasks, neural_model, DEVICE,
    max_tasks=100,
    dsl_time=30.0,
    ttt_steps=200 if DEVICE == 'cuda' else 20,
    n_ensemble=3 if DEVICE == 'cuda' else 1,
)

print('\n' + '='*60)
print('TRAINING SET RESULTS')
print('='*60)
for approach, data in train_results['approaches'].items():
    print(f'  {approach:20s}: {data["correct"]:3d}/{train_results["meta"]["n_tasks"]} = '
          f'{data["rate"]:.1%}  (avg cell acc: {data["avg_cell_acc"]:.3f})')

In [ ]:
# ============================================================
# Cell 10: Run Evaluation on EVAL Tasks (full ensemble)
# ============================================================
print('='*60)
print('EVALUATING ON EVAL TASKS (full ensemble)')
print('='*60)

eval_results = evaluate_all_approaches(
    eval_tasks, neural_model, DEVICE,
    max_tasks=None,
    dsl_time=30.0,
    ttt_steps=200 if DEVICE == 'cuda' else 20,
    n_ensemble=5 if DEVICE == 'cuda' else 1,
)

print('\n' + '='*60)
print('EVAL SET RESULTS')
print('='*60)
for approach, data in eval_results['approaches'].items():
    print(f'  {approach:20s}: {data["correct"]:3d}/{eval_results["meta"]["n_tasks"]} = '
          f'{data["rate"]:.1%}  (avg cell acc: {data["avg_cell_acc"]:.3f})')

In [ ]:
# ============================================================
# Cell 11: Analysis
# ============================================================

def analyze_results(results, label):
    print(f'\n{"="*60}')
    print(f'ANALYSIS: {label}')
    print(f'{"="*60}')

    tasks = results['per_task']

    print('\n--- Approach Overlap ---')
    dsl_set = {t['task_id'] for t in tasks if t['dsl_v2']['solved']}
    heur_set = {t['task_id'] for t in tasks if t['heuristic']['solved']}
    ttt_set = {t['task_id'] for t in tasks if t['neural_ttt']['solved']}
    ens_set = {t['task_id'] for t in tasks if t['neural_ensemble']['solved']}
    print(f'  DSL only:           {len(dsl_set - heur_set - ttt_set - ens_set)}')
    print(f'  Heuristic only:     {len(heur_set - dsl_set - ttt_set - ens_set)}')
    print(f'  Neural TTT only:    {len(ttt_set - dsl_set - heur_set - ens_set)}')
    print(f'  Ensemble only:      {len(ens_set - dsl_set - heur_set - ttt_set)}')
    print(f'  Ensemble bonus:     {len(ens_set - ttt_set)} (solved by ensemble but not single TTT)')
    all_solved = dsl_set | heur_set | ttt_set | ens_set
    print(f'  Any approach:       {len(all_solved)}/{len(tasks)} = {len(all_solved)/len(tasks):.1%}')

    print('\n--- By Shape Relationship ---')
    same = [t for t in tasks if t['same_shape']]
    diff = [t for t in tasks if not t['same_shape']]
    for label2, subset in [('Same shape', same), ('Diff shape', diff)]:
        n = len(subset)
        if n == 0: continue
        solved = sum(1 for t in subset if t['hybrid_best']['solved'])
        print(f'  {label2:12s}: {solved:3d}/{n} = {solved/n:.1%}')

    print('\n--- Cell Accuracy Distribution (Ensemble) ---')
    accs = [t['neural_ensemble']['cell_acc'] for t in tasks]
    if accs:
        aa = np.array(accs)
        print(f'  Mean: {np.mean(aa):.3f}')
        print(f'  Median: {np.median(aa):.3f}')
        print(f'  >95%: {np.sum(aa > 0.95)}/{len(aa)}')
        print(f'  >90%: {np.sum(aa > 0.9)}/{len(aa)}')
        print(f'  >80%: {np.sum(aa > 0.8)}/{len(aa)}')
        print(f'  >50%: {np.sum(aa > 0.5)}/{len(aa)}')

    print('\n--- DSL Solved ---')
    for t in [t for t in tasks if t['dsl_v2']['solved']][:15]:
        print(f'  {t["task_id"]}: {t["dsl_v2"]["program"]}')

    print('\n--- Heuristic Solved ---')
    for t in [t for t in tasks if t['heuristic']['solved']][:10]:
        print(f'  {t["task_id"]}: {t["heuristic"]["name"]}')

    print('\n--- Near-Misses (>90% cell acc, not solved) ---')
    near = sorted([t for t in tasks if t['neural_ensemble']['cell_acc'] > 0.9
                   and not t['neural_ensemble']['solved']],
                  key=lambda t: -t['neural_ensemble']['cell_acc'])
    for t in near[:15]:
        print(f'  {t["task_id"]}: {t["neural_ensemble"]["cell_acc"]:.1%}')

    print('\n--- Hard Tasks (all <50% cell acc) ---')
    hard = [t for t in tasks if t['hybrid_best']['cell_acc'] < 0.5]
    print(f'  {len(hard)}/{len(tasks)} ({len(hard)/len(tasks):.1%})')

    return {'dsl_set': dsl_set, 'heur_set': heur_set, 'ttt_set': ttt_set, 'ens_set': ens_set}

train_analysis = analyze_results(train_results, 'TRAINING')
eval_analysis = analyze_results(eval_results, 'EVAL')

In [ ]:
# ============================================================
# Cell 12: Timing Analysis â€” Can We Fit in 12 Hours?
# ============================================================

def timing_analysis(results, label):
    tasks = results['per_task']
    print(f'\n--- Timing: {label} ---')

    dsl_times = [t['dsl_v2']['time'] for t in tasks]
    heur_times = [t['heuristic']['time'] for t in tasks]
    no_ttt_times = [t['neural_no_ttt']['time'] for t in tasks]
    ttt_times = [t['neural_ttt']['time'] for t in tasks]

    print(f'  DSL avg:      {np.mean(dsl_times):.2f}s/task')
    print(f'  Heuristic avg: {np.mean(heur_times):.3f}s/task')
    print(f'  Neural no-TTT: {np.mean(no_ttt_times):.2f}s/task')
    print(f'  Neural TTT:    {np.mean(ttt_times):.2f}s/task')

    # Estimate for 400 test tasks in 12 hours
    n_test = 400  # ARC-AGI-2 test set
    budget = 12 * 3600  # 12 hours in seconds

    # Strategy: DSL first (fast), TTT only if DSL fails
    dsl_rate = results['approaches']['dsl_v2']['rate']
    dsl_avg = np.mean(dsl_times)
    ttt_avg = np.mean(ttt_times)

    dsl_time_total = n_test * dsl_avg
    ttt_tasks = int(n_test * (1 - dsl_rate))
    ttt_time_total = ttt_tasks * ttt_avg
    total = dsl_time_total + ttt_time_total

    print(f'\n  Estimated for {n_test} test tasks:')
    print(f'    DSL phase: {dsl_time_total/3600:.1f}h ({n_test} tasks)')
    print(f'    TTT phase: {ttt_time_total/3600:.1f}h ({ttt_tasks} tasks)')
    print(f'    Total:     {total/3600:.1f}h / 12h budget')
    print(f'    Fits:      {"YES" if total < budget else "NO â€” need to reduce TTT steps"}')

    if total > budget:
        max_ttt_per_task = (budget - dsl_time_total) / max(ttt_tasks, 1)
        print(f'    Max TTT time/task: {max_ttt_per_task:.1f}s')

timing_analysis(train_results, 'Training')
timing_analysis(eval_results, 'Eval')

In [ ]:
# ============================================================
# Cell 13: Save Results for Offline Analysis
# ============================================================

def to_json_safe(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.bool_,)): return bool(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, set): return sorted(list(obj))
    if isinstance(obj, dict): return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_json_safe(v) for v in obj]
    return obj

output = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'environment': 'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Local',
    'device': DEVICE,
    'gpu': torch.cuda.get_device_name() if DEVICE == 'cuda' else 'none',
    'model': {'zd': ZD, 'hd': HD, 'hid': HID,
              'params': sum(p.numel() for p in neural_model.parameters())},
    'training': to_json_safe(train_results),
    'eval': to_json_safe(eval_results),
    'summary': {
        'train_dsl': train_results['approaches']['dsl_v2']['rate'],
        'train_heur': train_results['approaches']['heuristic']['rate'],
        'train_ttt': train_results['approaches']['neural_ttt']['rate'],
        'train_ensemble': train_results['approaches']['neural_ensemble']['rate'],
        'train_hybrid': train_results['approaches']['hybrid_best']['rate'],
        'eval_dsl': eval_results['approaches']['dsl_v2']['rate'],
        'eval_heur': eval_results['approaches']['heuristic']['rate'],
        'eval_ttt': eval_results['approaches']['neural_ttt']['rate'],
        'eval_ensemble': eval_results['approaches']['neural_ensemble']['rate'],
        'eval_hybrid': eval_results['approaches']['hybrid_best']['rate'],
    }
}

out_path = 'arc_pathfinder_results.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f'\nResults saved to {out_path}')
print(f'File size: {os.path.getsize(out_path) / 1024:.1f} KB')

print('\n' + '='*60)
print('FINAL SUMMARY')
print('='*60)
s = output['summary']
print(f'                  TRAIN      EVAL')
print(f'  DSL v2:        {s["train_dsl"]:6.1%}    {s["eval_dsl"]:6.1%}')
print(f'  Heuristic:     {s["train_heur"]:6.1%}    {s["eval_heur"]:6.1%}')
print(f'  Neural TTT:    {s["train_ttt"]:6.1%}    {s["eval_ttt"]:6.1%}')
print(f'  Ensemble TTT:  {s["train_ensemble"]:6.1%}    {s["eval_ensemble"]:6.1%}')
print(f'  Hybrid best:   {s["train_hybrid"]:6.1%}    {s["eval_hybrid"]:6.1%}')

In [ ]:
# ============================================================
# Cell 14: Quick Visual Check â€” Show Some Predictions
# ============================================================

def show_task_prediction(task, prog_fns=None, neural_model=None, device='cpu'):
    """Text-based visualization of a task and prediction."""
    COLOR_MAP = '.123456789'
    def render(g):
        lines = []
        for r in range(g.shape[0]):
            lines.append(''.join(COLOR_MAP[int(v)] for v in g[r]))
        return lines

    print(f'\nTask: {task.task_id}')
    for i, p in enumerate(task.test[:1]):
        inp_lines = render(p.input)
        out_lines = render(p.output) if p.output is not None else ['?'*p.input.shape[1]]

        # Get prediction
        pred = None
        if prog_fns:
            pred = apply_prog(p.input, prog_fns)
        elif neural_model:
            try:
                pairs = [(pp.input, pp.output) for pp in task.train]
                preds = neural_model.solve(pairs, [p.input], device, refine=True, steps=30)
                pred = preds[0][0]
            except: pass

        pred_lines = render(pred) if pred is not None else ['ERR']

        max_h = max(len(inp_lines), len(out_lines), len(pred_lines))
        print(f'  {"Input":<{p.input.shape[1]+2}} {"Expected":<{p.output.shape[1]+2 if p.output is not None else 8}} Predicted')
        for r in range(max_h):
            i_row = inp_lines[r] if r < len(inp_lines) else ' '*p.input.shape[1]
            o_row = out_lines[r] if r < len(out_lines) else ' '*(p.output.shape[1] if p.output is not None else 1)
            p_row = pred_lines[r] if r < len(pred_lines) else ''
            match = ' ' if p.output is None else ('=' if r < len(out_lines) and r < len(pred_lines) and o_row == p_row else 'X')
            print(f'  {i_row}  {o_row}  {p_row} {match}')

# Show first few solved tasks
print('--- Sample DSL Predictions (Training) ---')
shown = 0
for tr in train_results['per_task']:
    if tr['dsl_v2']['solved'] and shown < 3:
        task = train_tasks[tr['task_id']]
        prog_result = search_dsl(task, max_depth=2, time_limit=10.0)
        if prog_result:
            _, fns = prog_result
            show_task_prediction(task, prog_fns=fns)
            shown += 1

# Show a few unsolved with neural prediction
print('\n--- Sample Neural TTT Predictions (Training, unsolved by DSL) ---')
shown = 0
for tr in train_results['per_task']:
    if not tr['dsl_v2']['solved'] and tr['neural_ttt']['cell_acc'] > 0.5 and shown < 3:
        task = train_tasks[tr['task_id']]
        show_task_prediction(task, neural_model=neural_model, device=DEVICE)
        shown += 1

---

## Interpretation Guide

### What to look for in `arc_pathfinder_results.json`:

1. **Overall rates**: `summary` section has all approach rates on train and eval
2. **Complementarity**: Do DSL and neural solve *different* tasks? If yes, hybrid has room to grow
3. **Cell accuracy**: Even when neural gets wrong answer, how close is it? High cell acc = architecture works, needs more training
4. **Near-misses**: Tasks with >80% cell acc but not solved â€” targeted fixes could flip these
5. **Hard tasks**: What fraction has <50% cell acc for ALL approaches? These need fundamentally new capabilities
6. **Timing**: Can the full pipeline fit in 12 hours on Kaggle?

### Paths forward based on results:

| If you see... | Then try... |
|---|---|
| DSL ~5-10%, Neural ~1-3% | More DSL primitives, bigger heuristic library |
| Neural cell acc >70% but exact match <5% | More TTT steps, augmentation, ensemble voting |
| DSL and Neural solve different tasks | Better routing/classifier to pick approach per task |
| Most tasks <50% cell acc | Need fundamentally new approach (LLM code gen, program synthesis) |
| Good train, bad eval | Overfitting; need more diverse training/augmentation |